# Run AtlasTailor on your own reference and target

A concise, executable template for AnnData inputs. The reference contains
the deeply measured source atlas. The target contributes spatial coordinates
and the measured panel at prediction time; full target expression is optional
and must be opened only after prediction locking when retrospective validation
is intended.

## Input contract

Both files require unique gene identifiers and finite 2D or 3D coordinates in
`obsm['spatial']` (or an explicitly supplied coordinate key). Counts belong in
`X` or a named layer. Panel design is source-only. At prediction time,
`read_target` uses backed access and materializes only the selected columns.

Set `ATLASTAILOR_REFERENCE` and `ATLASTAILOR_TARGET` before running. For a
prospective experiment, the target file may contain only the measured genes.
For retrospective masking, it may contain the full matrix because the loader
enforces column-restricted access.

In [ ]:
from __future__ import annotations

import hashlib
import os
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import hyperspatial as hs
from hyperspatial.io import read_reference, read_target, read_truth
from hyperspatial.metrics import evaluate_matrices

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = Path.cwd().parent


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(8 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def new_output(label: str) -> Path:
    root = Path(os.environ.get("ATLASTAILOR_RUNS", ROOT / "tutorial_runs"))
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    output = root / f"{label}_{stamp}"
    output.mkdir(parents=True, exist_ok=False)
    return output

REFERENCE_PATH = Path(os.environ["ATLASTAILOR_REFERENCE"])
TARGET_PATH = Path(os.environ["ATLASTAILOR_TARGET"])
COORDINATE_KEY = os.environ.get("ATLASTAILOR_COORDINATES", "spatial")

for path in (REFERENCE_PATH, TARGET_PATH):
    if not path.exists():
        raise FileNotFoundError(path)

{"reference": str(REFERENCE_PATH), "target": str(TARGET_PATH),
 "reference_sha256": sha256(REFERENCE_PATH), "target_sha256": sha256(TARGET_PATH)}

In [ ]:
run_root = new_output("atlastailor_user_data")
reference = read_reference(REFERENCE_PATH, coordinate_key=COORDINATE_KEY)
design = hs.design_panel(
    reference,
    config=hs.DesignConfig(budget=16, budgets=(8, 16, 32)),
    out=run_root / "panel_design",
)
assert len(design.genes) == 16
design.ordered_panel[["position", "gene"]]

In [ ]:
target_anchors = read_target(TARGET_PATH, design.genes, coordinate_key=COORDINATE_KEY)
assert target_anchors.measured_expression.shape[1] == 16
assert target_anchors.metadata["target_nonanchor_materialized"] is False

result = hs.adapt(
    reference,
    target_anchors,
    design.genes,
    out=run_root / "adaptation",
    config=hs.AdaptConfig(registration="geometry", seeds=(42, 43, 44)),
)
assert result.manifest.status == "completed"
assert result.manifest.data["target_nonanchor_access_before_prediction"] is False
hs.inspect_run(result.manifest.run_dir)

## Optional retrospective validation

Skip this section for prospective applications without target ground truth. If
the target file contains the full transcriptome, this cell opens it only after
the prediction manifest has completed.

In [ ]:
VALIDATE = os.environ.get("ATLASTAILOR_VALIDATE", "0") == "1"
if VALIDATE:
    truth, truth_coordinates = read_truth(TARGET_PATH, result.genes, coordinate_key=COORDINATE_KEY)
    np.testing.assert_allclose(truth_coordinates, result.coordinates)
    metrics = hs.validate(result, truth, out=run_root / "post_lock_metrics.tsv")
    display(metrics.head())
else:
    print("Prediction is locked. Set ATLASTAILOR_VALIDATE=1 only when retrospective truth is available.")